# 01 — Exploratory Data Analysis

Goal: understand the raw creditcard dataset before touching a model.
Outputs feed directly into feature engineering decisions in notebook 03.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/raw/creditcard.csv')
print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")

## 1. Basic structure

In [ ]:
print(df.dtypes)
df.head(3)

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

## 2. Class balance

In [ ]:
class_counts = df['Class'].value_counts()
fraud_rate = df['Class'].mean() * 100
print(f"Legitimate : {class_counts[0]:,}")
print(f"Fraud      : {class_counts[1]:,}")
print(f"Fraud rate : {fraud_rate:.4f}%")

fig, ax = plt.subplots(figsize=(4, 3))
class_counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'], edgecolor='white')
ax.set_xticklabels(['Legitimate', 'Fraud'], rotation=0)
ax.set_title('Class distribution (highly imbalanced)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/figures/01_class_balance.png')
plt.show()

## 3. Amount distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

df['Amount'].hist(bins=80, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Amount (raw)')
axes[0].set_xlabel('EUR')

np.log1p(df['Amount']).hist(bins=80, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Amount (log1p)')
axes[1].set_xlabel('log1p(EUR)')

plt.tight_layout()
plt.savefig('../reports/figures/01_amount_distribution.png')
plt.show()

print(f"Raw Amount  — mean: {df['Amount'].mean():.2f}, median: {df['Amount'].median():.2f}, max: {df['Amount'].max():.2f}")
print(f"Fraud mean  : {df[df['Class']==1]['Amount'].mean():.2f}")
print(f"Legit mean  : {df[df['Class']==0]['Amount'].mean():.2f}")

## 4. Time feature — hour-of-day fraud rate

In [ ]:
df['hour_of_day'] = (df['Time'] // 3600) % 24

hourly = df.groupby('hour_of_day')['Class'].mean() * 100

fig, ax = plt.subplots(figsize=(10, 3))
hourly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Fraud rate by hour of day')
ax.set_xlabel('Hour')
ax.set_ylabel('Fraud rate (%)')
plt.tight_layout()
plt.savefig('../reports/figures/01_fraud_by_hour.png')
plt.show()

## 5. Feature correlations with Class

In [ ]:
correlations = df.drop(columns=['Time', 'hour_of_day']).corr()['Class'].drop('Class')
correlations_sorted = correlations.abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
correlations_sorted.head(15).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 15 features by absolute correlation with Class')
ax.set_ylabel('|correlation|')
plt.tight_layout()
plt.savefig('../reports/figures/01_feature_correlations.png')
plt.show()

print("Top 10 features correlated with fraud:")
print(correlations_sorted.head(10).to_string())

## 6. PCA feature separation — top 3 vs Class

In [ ]:
top3 = correlations_sorted.head(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, feat in zip(axes, top3):
    df[df['Class']==0][feat].hist(bins=60, ax=ax, alpha=0.6, label='Legit', color='steelblue', density=True)
    df[df['Class']==1][feat].hist(bins=60, ax=ax, alpha=0.6, label='Fraud', color='tomato', density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)
plt.suptitle('Distribution of top-3 fraud signals', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/01_top3_features.png')
plt.show()

## 7. EDA summary

In [ ]:
print("=" * 55)
print("EDA SUMMARY")
print("=" * 55)
print(f"\n1. CLASS IMBALANCE")
print(f"   Fraud rate: {fraud_rate:.4f}% — use class_weight or SMOTE.")
print(f"   Accuracy is a useless metric here; use PR-AUC.")
print(f"\n2. AMOUNT")
print(f"   Median: {df['Amount'].median():.2f}")
print(f"   Amount needs log scaling before modelling.")
print(f"\n3. TIME")
print(f"   Fraud rate varies by hour - time is a useful feature.")
print(f"   Convert raw seconds to hour-of-day.")
print(f"\n4. FEATURES")
top3 = correlations_sorted.head(3).index.tolist()
print(f"   Strongest fraud signals: {top3}")
print(f"   These PCA features show clear class separation.")
print(f"\n5. MISSING DATA")
print(f"   {df.isnull().sum().sum()} missing values - no imputation needed.")
print(f"\nNext step: notebook 02 - build a baseline model using these insights.")
print("=" * 55)